##St. Valentine's Hackathon 
Entry: Parth Sinha

In [22]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer
import lightgbm as lgb
import catboost as cb

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [23]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

TARGET = 'Has_Valentine'
ID_COL = 'Id'

y = train[TARGET].values
test_ids = test[ID_COL].values

In [24]:
def feature_engineer(df):
    df = df.copy()

    # Parsing date features
    df['Survey_Date'] = pd.to_datetime(df['Survey_Date'], errors='coerce')
    df['survey_month']      = df['Survey_Date'].dt.month
    df['survey_dayofweek']  = df['Survey_Date'].dt.dayofweek
    df['survey_hour']       = df['Survey_Date'].dt.hour
    df['survey_quarter']    = df['Survey_Date'].dt.quarter
    df['is_weekend']        = (df['survey_dayofweek'] >= 5).astype(int)
    df['is_feb']            = (df['survey_month'] == 2).astype(int)  # valentines month!
    df.drop(columns=['Survey_Date'], inplace=True)

    # BMI
    df['BMI'] = df['Weight_kg'] / ((df['Height_cm'] / 100) ** 2)

    # Interaction features
    df['income_per_age']           = df['Income'] / (df['Age'] + 1)
    df['appearance_x_social']      = df['Appearance_Score'] * df['Social_Skills_Score']
    df['appearance_x_extraversion']= df['Appearance_Score'] * df['Extraversion_Score']
    df['social_x_extraversion']    = df['Social_Skills_Score'] * df['Extraversion_Score']
    df['app_user_x_social']        = df['Dating_App_User'] * df['Social_Skills_Score']
    df['app_user_x_appearance']    = df['Dating_App_User'] * df['Appearance_Score']
    df['prev_rel_x_extraversion']  = df['Previous_Relationships'] * df['Extraversion_Score']
    df['score_sum']                = df['Appearance_Score'].fillna(0) + df['Social_Skills_Score'].fillna(0) + df['Extraversion_Score'].fillna(0)
    df['score_mean']               = df['score_sum'] / 3
    df['income_log']               = np.log1p(df['Income'].clip(lower=0))
    df['age_bin']                  = pd.cut(df['Age'], bins=[0,25,35,45,55,100], labels=False)

    return df

In [25]:
train = feature_engineer(train)
test  = feature_engineer(test)

In [26]:
train.head()

,Id,Age,Gender,Income,Appearance_Score,Education,Job_Type,Social_Media_Presence,Social_Skills_Score,Extraversion_Score,...,appearance_x_social,appearance_x_extraversion,social_x_extraversion,app_user_x_social,app_user_x_appearance,prev_rel_x_extraversion,score_sum,score_mean,income_log,age_bin
0,1,30.0,Female,53669.24,4.33,NaN,Senior,Low,9.64,6.51,...,41.7412,28.1883,62.7564,0.00,0.00,19.53,20.48,6.826667,10.890614,1.0
1,2,51.0,Female,40939.50,4.22,High School,Mid Level,Medium,1.93,3.57,...,8.1446,15.0654,6.8901,1.93,4.22,10.71,9.72,3.240000,10.619875,3.0
2,3,57.0,Female,28554.52,NaN,Bachelor,Senior,Medium,8.82,1.01,...,NaN,NaN,8.9082,0.00,NaN,0.00,9.83,3.276667,10.259606,4.0
3,4,26.0,Male,28121.82,5.59,Bachelor,Mid Level,Low,4.73,2.56,...,26.4407,14.3104,12.1088,NaN,NaN,5.12,12.88,4.293333,10.244337,1.0
4,5,59.0,Male,42020.75,5.71,Bachelor,Entry Level,Low,3.87,3.14,...,22.0977,17.9294,12.1518,3.87,5.71,NaN,12.72,4.240000,10.645943,4.0


In [27]:
train[['income_log']]

,income_log
0,10.890614
1,10.619875
2,10.259606
3,10.244337
4,10.645943
...,...
699995,11.120536
699996,10.270368
699997,11.124787
699998,9.615872


In [28]:
train[['Weight_kg','Height_cm','BMI','income_log','Has_Valentine']].corr()

,Weight_kg,Height_cm,BMI,income_log,Has_Valentine
Weight_kg,1.000000,-0.001235,0.851694,0.000774,0.000200
Height_cm,-0.001235,1.000000,-0.511168,-0.003223,0.001160
BMI,0.851694,-0.511168,1.000000,0.003174,-0.000112
income_log,0.000774,-0.003223,0.003174,1.000000,0.042605
Has_Valentine,0.000200,0.001160,-0.000112,0.042605,1.000000


In [19]:
train['income_log']

0         10.890614
1         10.619875
2         10.259606
3         10.244337
4         10.645943
            ...    
699995    11.120536
699996    10.270368
699997    11.124787
699998     9.615872
699999    11.004763
Name: income_log, Length: 700000, dtype: float64

In [6]:
train.groupby('Has_Valentine')['BMI'].mean()

Has_Valentine
0    26.242346
1    26.240980
Name: BMI, dtype: float64

In [7]:
train[['Has_Valentine', 'survey_month']].corr()

,Has_Valentine,survey_month
Has_Valentine,1.000000,0.000935
survey_month,0.000935,1.000000


In [8]:
train['Has_Valentine'].mean()

0.49791857142857143

In [9]:
CAT_COLS = ['Gender', 'Education', 'Job_Type', 'Social_Media_Presence',
            'Location_Type', 'Zodiac_Sign', 'Pets', 'Favorite_Color']

all_df = pd.concat([train, test], axis=0).reset_index(drop=True)

for col in CAT_COLS:
    all_df[col] = all_df[col].fillna('MISSING')
    le = LabelEncoder()
    all_df[col] = le.fit_transform(all_df[col].astype(str))

train_enc = all_df.iloc[:len(train)].copy()
test_enc  = all_df.iloc[len(train):].copy()

DROP_COLS = [ID_COL, TARGET] if TARGET in train_enc.columns else [ID_COL]
FEATURE_COLS = [c for c in train_enc.columns if c not in DROP_COLS and c != TARGET]

X_train = train_enc[FEATURE_COLS].values.astype(np.float32)
X_test  = test_enc[FEATURE_COLS].values.astype(np.float32)

In [10]:
imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(X_train)
X_test  = imputer.transform(X_test)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train).astype(np.float32)
X_test_sc  = scaler.transform(X_test).astype(np.float32)

In [11]:
#check_df = pd.DataFrame(X_train_sc, columns = FEATURE_COLS)
#check_df.head()

,Age,Gender,Income,Appearance_Score,Education,Job_Type,Social_Media_Presence,Social_Skills_Score,Extraversion_Score,Dating_App_User,...,appearance_x_social,appearance_x_extraversion,social_x_extraversion,app_user_x_social,app_user_x_appearance,prev_rel_x_extraversion,score_sum,score_mean,income_log,age_bin
0,-0.832749,-1.034910,0.126926,-0.525659,0.829095,0.749335,-0.738636,2.152755,0.796389,-1.290640,...,1.043801,0.253855,2.506379,-1.333484,-1.221578,1.227014,1.525343,1.525343,0.525245,-0.781478
1,0.756770,-1.034910,-0.177099,-0.604238,0.131569,-0.128943,1.178050,-2.394234,-0.765013,0.774809,...,-1.926906,-0.863516,-1.736190,-0.642588,0.431070,0.121671,-0.988352,-0.988353,0.121886,0.720292
2,1.210918,-1.034910,-0.472890,-0.011328,-0.565956,0.749335,1.178050,1.669158,-2.124600,-1.290640,...,-0.085111,-0.106622,-1.582932,-1.333484,0.211762,-1.220531,-0.962655,-0.962655,-0.414861,1.471177
3,-1.135514,0.958306,-0.483224,0.374420,-0.565956,-0.128943,-0.738636,-0.742928,-1.301412,0.774809,...,-0.309113,-0.927802,-1.339874,0.119902,0.211762,-0.578881,-0.250130,-0.250130,-0.437608,-0.781478
4,1.362301,0.958306,-0.151275,0.460142,-0.565956,-1.446360,-0.738636,-1.250115,-0.993381,0.774809,...,-0.693133,-0.619656,-1.336608,0.051887,1.014589,-0.222966,-0.287508,-0.287508,0.160723,1.471177


In [16]:
#check_df[['Age', 'Appearance_Score', 'Education', 'app_user_x_appearance']].median

<bound method DataFrame.median of              Age  Appearance_Score  Education  app_user_x_appearance
0      -0.832749         -0.525659   0.829095              -1.221578
1       0.756770         -0.604238   0.131569               0.431070
2       1.210918         -0.011328  -0.565956               0.211762
3      -1.135514          0.374420  -0.565956               0.211762
4       1.362301          0.460142  -0.565956               1.014589
...          ...               ...        ...                    ...
699995 -1.438280          2.396026   1.526621              -1.221578
699996 -0.151526          0.653016  -0.565956              -1.221578
699997 -1.589662         -0.275637  -1.263482               0.611217
699998 -1.059823          0.174402  -0.565956               0.857939
699999  0.756770         -0.739964  -0.565956              -1.221578

[700000 rows x 4 columns]>

In [20]:
class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(x + self.block(x))

In [21]:
class DeepTabNet(nn.Module):
    def __init__(self, input_dim, hidden_dims=[512, 512, 256, 256, 128], dropout=0.3):
        super().__init__()
        layers = [nn.Linear(input_dim, hidden_dims[0]),
                  nn.BatchNorm1d(hidden_dims[0]),
                  nn.GELU(),
                  nn.Dropout(dropout)]
        for i in range(len(hidden_dims) - 1):
            layers += [nn.Linear(hidden_dims[i], hidden_dims[i+1]),
                       nn.BatchNorm1d(hidden_dims[i+1]),
                       nn.GELU(),
                       nn.Dropout(dropout)]
            if hidden_dims[i] == hidden_dims[i+1]:
                layers.append(ResidualBlock(hidden_dims[i+1], dropout))
        layers += [nn.Linear(hidden_dims[-1], 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)

In [22]:
def train_nn_fold(X_tr, y_tr, X_val, y_val, input_dim, device, epochs=100, lr=5e-4, batch_size=2048):
    model = DeepTabNet(input_dim).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.BCEWithLogitsLoss()

    # class weight for imbalance
    pos_weight = torch.tensor([(y_tr == 0).sum() / (y_tr == 1).sum()]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    X_tr_t  = torch.tensor(X_tr, dtype=torch.float32)
    y_tr_t  = torch.tensor(y_tr, dtype=torch.float32)
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.float32)

    train_ds = TensorDataset(X_tr_t, y_tr_t)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)

    best_auc   = 0
    best_preds = None
    patience   = 15
    wait       = 0

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t.to(device)).cpu().numpy()
        val_preds = torch.sigmoid(torch.tensor(val_logits)).numpy()
        auc = roc_auc_score(y_val, val_preds)

        if auc > best_auc:
            best_auc   = auc
            best_preds = val_preds.copy()
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    # Final test preds
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        test_preds = torch.sigmoid(model(torch.tensor(X_test_sc).to(device))).cpu().numpy()

    return best_preds, test_preds, best_auc

In [23]:
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

input_dim = X_train_sc.shape[1]

oof_nn  = np.zeros(len(y))
oof_lgb = np.zeros(len(y))
oof_cb  = np.zeros(len(y))

test_nn_preds  = np.zeros(len(X_test_sc))
test_lgb_preds = np.zeros(len(X_test_sc))
test_cb_preds  = np.zeros(len(X_test_sc))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_sc, y)):
    print(f"\n{'='*40}\nFold {fold+1}/{N_SPLITS}\n{'='*40}")

    X_tr_sc,  X_val_sc  = X_train_sc[tr_idx],  X_train_sc[val_idx]
    X_tr_raw, X_val_raw = X_train[tr_idx],      X_train[val_idx]
    y_tr,     y_val     = y[tr_idx],             y[val_idx]

    # ── Neural Net ──────────────────────────
    val_p_nn, test_p_nn, nn_auc = train_nn_fold(
        X_tr_sc, y_tr, X_val_sc, y_val, input_dim, device
    )
    oof_nn[val_idx]  = val_p_nn
    test_nn_preds   += test_p_nn / N_SPLITS
    print(f"  NN  AUC: {nn_auc:.5f}")

    # ── LightGBM ────────────────────────────
    lgb_params = dict(
        objective='binary', metric='auc', verbosity=-1,
        learning_rate=0.02, num_leaves=127, max_depth=-1,
        min_child_samples=20, feature_fraction=0.8,
        bagging_fraction=0.8, bagging_freq=5,
        reg_alpha=0.1, reg_lambda=1.0,
        n_estimators=3000, random_state=42,
    )
    lgb_model = lgb.LGBMClassifier(**lgb_params)
    lgb_model.fit(
        X_tr_raw, y_tr,
        eval_set=[(X_val_raw, y_val)],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)]
    )
    val_p_lgb = lgb_model.predict_proba(X_val_raw)[:, 1]
    oof_lgb[val_idx]  = val_p_lgb
    test_lgb_preds += lgb_model.predict_proba(X_test)[:, 1] / N_SPLITS
    lgb_auc = roc_auc_score(y_val, val_p_lgb)
    print(f"  LGB AUC: {lgb_auc:.5f}")

    # ── CatBoost ────────────────────────────
    cb_model = cb.CatBoostClassifier(
        iterations=3000, learning_rate=0.02, depth=7,
        eval_metric='AUC', loss_function='Logloss',
        l2_leaf_reg=3, random_strength=1,
        bagging_temperature=0.5, od_type='Iter', od_wait=100,
        random_seed=42, verbose=0,
    )
    cb_model.fit(
        X_tr_raw, y_tr,
        eval_set=(X_val_raw, y_val),
        use_best_model=True
    )
    val_p_cb = cb_model.predict_proba(X_val_raw)[:, 1]
    oof_cb[val_idx]  = val_p_cb
    test_cb_preds   += cb_model.predict_proba(X_test)[:, 1] / N_SPLITS
    cb_auc = roc_auc_score(y_val, val_p_cb)
    print(f"  CB  AUC: {cb_auc:.5f}")

Device: cuda

Fold 1/5
  NN  AUC: 0.59004
  LGB AUC: 0.59052
  CB  AUC: 0.59144

Fold 2/5
  NN  AUC: 0.58978
  LGB AUC: 0.58987
  CB  AUC: 0.59086

Fold 3/5
  NN  AUC: 0.59150
  LGB AUC: 0.59180
  CB  AUC: 0.59319

Fold 4/5
  NN  AUC: 0.59469
  LGB AUC: 0.59424
  CB  AUC: 0.59621

Fold 5/5
  NN  AUC: 0.59421
  LGB AUC: 0.59496
  CB  AUC: 0.59578


In [24]:
from scipy.optimize import minimize

def neg_auc(weights, oofs):
    blend = sum(w * o for w, o in zip(weights, oofs))
    return -roc_auc_score(y, blend)

result = minimize(
    neg_auc,
    x0=[1/3, 1/3, 1/3],
    args=([oof_nn, oof_lgb, oof_cb],),
    method='Nelder-Mead',
    options={'maxiter': 2000, 'xatol': 1e-6}
)
w_nn, w_lgb, w_cb = result.x
print(f"\nOptimal weights — NN: {w_nn:.4f}, LGB: {w_lgb:.4f}, CB: {w_cb:.4f}")

oof_blend = w_nn * oof_nn + w_lgb * oof_lgb + w_cb * oof_cb
final_auc = roc_auc_score(y, oof_blend)
print(f"Final blended OOF AUC: {final_auc:.5f}")

# Print individual OOF scores
print(f"NN  OOF AUC: {roc_auc_score(y, oof_nn):.5f}")
print(f"LGB OOF AUC: {roc_auc_score(y, oof_lgb):.5f}")
print(f"CB  OOF AUC: {roc_auc_score(y, oof_cb):.5f}")


Optimal weights — NN: 0.1919, LGB: 0.0201, CB: 0.7487
Final blended OOF AUC: 0.59361
NN  OOF AUC: 0.59182
LGB OOF AUC: 0.59226
CB  OOF AUC: 0.59349


In [25]:
test_blend = w_nn * test_nn_preds + w_lgb * test_lgb_preds + w_cb * test_cb_preds

submission = pd.DataFrame({
    'Id':            test_ids,
    'Has_Valentine': test_blend
})
submission.to_csv('submission.csv', index=False)
print("\nSaved submission.csv")
print(submission.head())


Saved submission.csv
       Id  Has_Valentine
0  700001       0.466321
1  700002       0.424413
2  700003       0.429409
3  700004       0.425124
4  700005       0.484104
